# `aidp-exacs` live test — plain user/password on TCP 1521 + server-enforced NNE
**Live-test row 6.** Mirrors the working pattern from `exacs_intransit_encryption_demo.ipynb` in the `exacs-private-test` workspace, which proved AES256 NNE end-to-end against a customer ExaCS cluster running Oracle 23ai.

**Workspace prereq:** `networkConfigurationDetails.scanDetails` must include the SCAN FQDN+port (PE-ARCH 3c, RCE with SCAN Proxy). Without it the RAC redirect dies with ORA-17820.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


## Step 1 — Configuration


In [ ]:
SCAN_HOST = os.environ['EXACS_HOST']
SCAN_PORT = int(os.environ.get('EXACS_PORT_LEGACY', '1521'))
SERVICE   = os.environ['EXACS_SERVICE_NAME']
DB_USER   = os.environ['EXACS_USER']
DB_PASS   = os.environ['EXACS_PASSWORD']
TABLE     = os.environ['EXACS_TABLE_FOR_TEST']
print(f'  SCAN  : {SCAN_HOST}:{SCAN_PORT}')
print(f'  SVC   : {SERVICE}')
print(f'  USER  : {DB_USER}')
print(f'  TABLE : {TABLE}')


## Step 2 — DNS resolution check
SCAN host must resolve to a Class-E (240.0.0.0/4 / 255.x) or RFC-1918 (10.x/172.16-31.x/192.168.x) IP. A public IP would mean traffic isn't going through the PE.


In [ ]:
import socket, ipaddress
ips = sorted({r[4][0] for r in socket.getaddrinfo(SCAN_HOST, SCAN_PORT, socket.AF_INET)})
for ip in ips:
    is_class_e = int(ip.split('.')[0]) >= 240
    is_priv    = ipaddress.ip_address(ip).is_private
    kind = 'Class-E NAT' if is_class_e else 'RFC-1918 private' if is_priv else 'PUBLIC (FAIL)'
    print(f'  {SCAN_HOST} -> {ip}  [{kind}]')
assert any(int(ip.split('.')[0]) >= 240 or ipaddress.ip_address(ip).is_private for ip in ips), 'PE routing not active'


## Step 3 — TCP connectivity


In [ ]:
import socket, time
t0 = time.time()
with socket.create_connection((SCAN_HOST, SCAN_PORT), timeout=15) as s:
    print(f'  Connected in {(time.time()-t0)*1000:.0f} ms  (local={s.getsockname()}  remote={s.getpeername()})')


## Step 4 — Spark JDBC connect (plain user/password)


In [ ]:
from oracle_ai_data_platform_connectors.jdbc import (
    build_oracle_jdbc_url, spark_jdbc_options_password,
)
url = build_oracle_jdbc_url(host=SCAN_HOST, port=SCAN_PORT, service_name=SERVICE, use_tcps=False)
opts = spark_jdbc_options_password(url=url, user=DB_USER, password=DB_PASS)
# Optional: enable for the SYS user only.
# opts['oracle.jdbc.internal_logon'] = 'sysdba'
print('JDBC URL:', url)


In [ ]:
df = (spark.read.format('jdbc').options(**opts).option('dbtable', TABLE).load())
df.show(5)


## Step 5 — In-transit encryption verification
Read `v$session_connect_info.network_service_banner` for the active session — proves AES256 NNE.


In [ ]:
enc_q = (
    "SELECT network_service_banner FROM v$session_connect_info "
    "WHERE sid = SYS_CONTEXT('USERENV','SID')"
)
banners = (spark.read.format('jdbc').options(**opts)
             .option('query', enc_q).load().collect())
for r in banners:
    print(' ', r[0])
encryption = next((b[0] for b in (b.split(' Encryption service adapter') for b in (r[0] for r in banners)) if len(b) > 1), None)
print('Negotiated algorithm:', encryption)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-exacs',
    'auth': 'password-tcp-nne',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
